# Pra-pemrosesan data harga harian per pasar 

Membaca tab harian dari `data/raw/xlsx_asli` (lewat `_inventaris_tab.csv`), mengubahnya ke format panjang,
menyeragamkan nama dan satuan, lalu mengekspor tabel staging untuk dimuat ke PostgreSQL.

**Keluaran**: `data/staging/stg_harga_harian.csv` (10 komoditas terpilih), `stg_hari_tanpa_data.csv`; `data/processed/harga_pasar_semua_komoditas.csv`, `seleksi_komoditas.csv`, `harga_komoditas.csv`.

Aturan penting: `-` dan `0` = tidak ada harga (NULL), bukan harga nol.

In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..")
RAW = ROOT / "data" / "raw"
STG = ROOT / "data" / "staging"
STG.mkdir(parents=True, exist_ok=True)

inv = pd.read_csv(RAW / "_inventaris_tab.csv", parse_dates=["tanggal"], dtype={"tab": str})
print(f"Tab di inventaris: {len(inv):,} | nomor di luar bulan (dilewati): {inv['tanggal'].isna().sum()}")

Tab di inventaris: 1,336 | nomor di luar bulan (dilewati): 1


## 1. Parser satu tab
Kolom pasar, satuan, dan Rata-Rata dikenali dari teks header, bukan posisi kolom.
Baris tanpa satuan yang namanya huruf kapital semua, atau yang tidak punya nilai di kolom pasar, dianggap judul kelompok.

In [3]:
def parse_tab(xl, tab):
    """Satu tab harian -> format panjang. Mengembalikan None bila tab kosong."""
    nama = next(s for s in xl.sheet_names if s.strip() == str(tab))
    raw = pd.read_excel(xl, sheet_name=nama, header=None)
    if raw.dropna(how="all").empty:
        return None
    txt = raw.map(lambda v: (re.sub(r"\s+", " ", v).strip() or float("nan")) if isinstance(v, str) else v)

    h = next((i for i in range(min(15, len(txt)))
              if txt.iloc[i].astype(str).str.upper().str.contains("NAMA BAHAN").any()), None)
    if h is None:
        raise ValueError("header 'NAMA BAHAN' tidak ditemukan")
    head = txt.iloc[h]
    pasar = {c: re.sub(r"(?i)^pasar\s+", "", v) for c, v in head.items()
             if isinstance(v, str) and v.lower().startswith("pasar")}
    if not pasar:
        raise ValueError("kolom pasar tidak ditemukan")
    c_sat = next((c for c, v in head.items() if isinstance(v, str) and v.upper() == "SATUAN"), None)
    c_avg = next((c for c, v in head.items() if isinstance(v, str) and v.lower().startswith("rata")), None)
    kiri = [c for c in txt.columns if c < min(pasar) and c != c_sat]
    kolom_pasar = list(pasar)

    rec, grup = [], None
    for i in range(h + 1, len(txt)):
        r = txt.iloc[i]
        nm = next((r[c] for c in kiri if isinstance(r[c], str) and not r[c].isdigit()), None)
        if nm is None:
            continue
        sat = r[c_sat] if c_sat is not None and isinstance(r[c_sat], str) else None
        if sat is None and (nm.isupper() or not r.loc[kolom_pasar].notna().any()):
            grup = nm                                    # baris judul kelompok
            continue
        harga = pd.to_numeric(r.loc[kolom_pasar], errors="coerce")
        avg = pd.to_numeric(r[c_avg], errors="coerce") if c_avg is not None else float("nan")
        for c, p in pasar.items():
            rec.append((grup, re.sub(r"^[-\s]+", "", nm), sat, p, harga[c], avg))

    out = pd.DataFrame(rec, columns=["kelompok", "komoditas", "satuan", "pasar", "harga", "rata_sheet"])
    out["harga"] = out["harga"].where(out["harga"] > 0)   # "-" / 0 = tidak ada harga
    return out

## 2. Jalankan ke semua tab
Tiap file dibuka sekali. Hasil per tab dicatat di `_parse_log.csv`.

In [4]:
hasil, log = [], []
for file, g in inv.dropna(subset=["tanggal"]).groupby("file", sort=False):
    xl = pd.ExcelFile(g["path"].iloc[0])
    for _, r in g.iterrows():
        try:
            d = parse_tab(xl, r["tab"])
        except Exception as e:
            log.append((file, r["tab"], r["tanggal"], f"ERROR: {e!r}", 0))
            continue
        if d is None:
            log.append((file, r["tab"], r["tanggal"], "KOSONG", 0))
        else:
            hasil.append(d.assign(tanggal=r["tanggal"], file=file))
            log.append((file, r["tab"], r["tanggal"], "OK", len(d)))
    xl.close()
    print(".", end="")

fakta = pd.concat(hasil, ignore_index=True)
fakta["kelompok"] = fakta["kelompok"].str.replace(r"\b(\w) (?=\w\b)", r"\1", regex=True)   # "I K A N" -> "IKAN"

df_log = pd.DataFrame(log, columns=["file", "tab", "tanggal", "status", "baris"])
df_log.to_csv(RAW / "_parse_log.csv", index=False)
print("\n", df_log["status"].str.split(":").str[0].value_counts().to_string())
print(f"Baris: {len(fakta):,} | tanggal: {fakta['tanggal'].nunique()} | pasar: {fakta['pasar'].nunique()}")
df_log[df_log["status"].str.startswith("ERROR")]

............................................
 status
OK        1332
KOSONG       3
Baris: 601,048 | tanggal: 1332 | pasar: 7


,file,tab,tanggal,status,baris


## 3. Hari tanpa data
Hari kalender yang tidak punya baris harga: tab tidak ada (mis. hari libur Idulfitri/Iduladha) atau tab kosong.

In [5]:
semua = pd.date_range(inv["tanggal"].min().replace(day=1), inv["tanggal"].max() + pd.offsets.MonthEnd(0))
tanpa = pd.DataFrame({"tanggal": semua.difference(pd.DatetimeIndex(fakta["tanggal"].unique()))})
tanpa["alasan"] = np.where(tanpa["tanggal"].isin(df_log.loc[df_log["status"] == "KOSONG", "tanggal"]),
                           "tab kosong", "tidak ada tab")
print(f"Hari kalender: {len(semua):,} | hari tanpa data: {len(tanpa)}")
tanpa

Hari kalender: 1,339 | hari tanpa data: 7


,tanggal,alasan
0,2023-04-22,tidak ada tab
1,2023-04-23,tidak ada tab
2,2023-06-29,tidak ada tab
3,2025-03-31,tidak ada tab
4,2026-04-28,tab kosong
5,2026-04-29,tab kosong
6,2026-04-30,tab kosong


## 4. Validasi: rata-rata 7 pasar vs kolom Rata-Rata di sheet
Selisih yang tersisa seharusnya hanya kasus saat sheet ikut menghitung pasar bernilai `0` (kolom `tambahan` = jumlah pasar
ekstra yang dihitung sheet). Data kita memperlakukan 0 sebagai tidak ada harga, jadi rata-rata kita yang benar.

In [6]:
c = (fakta.groupby(["file", "tanggal", "komoditas"])
          .agg(rata=("harga", "mean"), n_ada=("harga", "count"), total=("harga", "sum"), sheet=("rata_sheet", "first")))
c = c[c["sheet"] > 0].copy()
c["selisih"] = (c["rata"].fillna(0) - c["sheet"]).abs()
bad = c[c["selisih"] > 1].copy()                                   # toleransi Rp 1
bad["tambahan"] = (bad["total"] / bad["sheet"] - bad["n_ada"]).round(2)

print(f"Dicek: {len(c):,} komoditas-hari | selisih > Rp1: {len(bad):,}")
print("Pasar ekstra yang dihitung sheet:", bad["tambahan"].value_counts().to_dict())
print(bad.groupby("file").size().sort_values(ascending=False).head(10).to_string())

Dicek: 85,358 komoditas-hari | selisih > Rp1: 174
Pasar ekstra yang dihitung sheet: {1.0: 167, 2.0: 7}
file
06. REKAPITULASI JUNI 2026.xlsx        66
03. REKAPITULASI MARET 2025...xlsx     30
02. REKAPITULASI FEBRUARI 2025.xlsx    28
04. REKAPITULASI APRIL 2025.xlsx       23
01. REKAPITULASI JANUARI 2025.xlsx     17
02. REKAP FEBRUARI 2023.xlsx           10


## 5. Pemetaan nama baku, kelompok, dan satuan
- Beras: Kwalitas I → Premium, II → Medium Bulog, III dan Medium → Medium Non Bulog.
- Varian ejaan disatukan; kelompok ditentukan dari nama komoditas (kelompok paling sering), bukan posisi baris.
- Tahu dan Tempe digabung ke satu kelompok. Satuan `Kg`/`kg` diseragamkan.
- Garam Halus berganti 250 gr → 500 gr (21 Jan 2024): `harga_baku` disetarakan ke 500 gr, `harga` tetap apa adanya.

In [7]:
ALIAS = {   # kunci = nama casefold; nilai = nama baku (casefold)
    'susu kental manis "bendera"': 'susu kental manis "bendera" putih',
    "beras ir.64 kwalitas i":   "beras premium",
    "beras ir.64 kwalitas ii":  "beras medium bulog",
    "beras ir.64 kwalitas iii": "beras medium non bulog",
    "beras ir.64 premium":      "beras premium",
    "beras ir.64 medium bulog": "beras medium bulog",
    "beras ir.64 medium":       "beras medium non bulog",
    "beras medium":             "beras medium non bulog",
}
NAMA_BAKU = {   # nama tampilan, sama dengan repo lama
    "beras premium": "Beras Premium",
    "beras medium bulog": "Beras Medium Bulog",
    "beras medium non bulog": "Beras Medium Non Bulog",
    "gula pasir lokal curah": "Gula Pasir Lokal Curah",
    "minyak kita": "Minyak Kita",
    'susu kental manis "bendera" putih': 'Susu Kental Manis "Bendera" Putih',
}
KEL_ALIAS = {"TAHU": "TAHU DAN TEMPE", "TEMPE": "TAHU DAN TEMPE"}
modus = lambda s: s.value_counts().index[0]

fakta["satuan"] = (fakta["satuan"].str.strip().str.replace(r"(?i)^kg$", "Kg", regex=True)
                   .replace({"400 Gr/Dos": "400 Gr/Dus"}))
fakta["kunci"] = (fakta["komoditas"].str.replace(r"\s+", " ", regex=True)
                  .str.strip().str.casefold().replace(ALIAS))
fakta["komoditas_baku"] = fakta["kunci"].map(fakta.groupby("kunci")["komoditas"].agg(modus))
fakta["komoditas_baku"] = fakta["kunci"].map(NAMA_BAKU).fillna(fakta["komoditas_baku"])
fakta["kelompok_baku"] = (fakta["komoditas_baku"]
                          .map(fakta.groupby("komoditas_baku")["kelompok"].agg(modus))
                          .replace(KEL_ALIAS))

# garam: samakan ke 500 gr, harga asli tetap disimpan
fakta["satuan_baku"] = fakta["satuan"].replace({"250 Gr/Bngks": "500 Gr/Bngks"})
fakta["harga_baku"] = fakta["harga"].where(fakta["satuan"] != "250 Gr/Bngks", fakta["harga"] * 2)

print("Baris ganda (tanggal+pasar+komoditas):", fakta.duplicated(["tanggal", "pasar", "komoditas_baku"], keep=False).sum())
print("Kelompok:", fakta["kelompok_baku"].nunique(), "| Komoditas baku:", fakta["komoditas_baku"].nunique())
sat = fakta.dropna(subset=["harga"]).groupby("komoditas_baku")["satuan"].agg(lambda s: sorted(s.dropna().unique()))
print("Komoditas dengan lebih dari satu satuan:", sat[sat.map(len) > 1].to_dict())

Baris ganda (tanggal+pasar+komoditas): 0
Kelompok: 22 | Komoditas baku: 70
Komoditas dengan lebih dari satu satuan: {'Garam Halus (Cap Kapal)': ['250 Gr/Bngks', '500 Gr/Bngks']}


In [8]:
# --- Koreksi salah ketik (sudah diverifikasi ke data sumber) ---
def baris(rentang, pasar, komoditas, asli, benar, alasan):
    return [(t, pasar, komoditas, asli, benar, alasan) for t in pd.date_range(*rentang)]

KOREKSI = pd.DataFrame(
    baris(("2024-07-15", "2024-07-15"), "Pabean",      "Telur Ayam Kampung", 23000,  2300,  "kelebihan satu nol") +
    baris(("2025-06-15", "2025-06-17"), "Pucang Anom", "Tomat",             250000, 25000, "kelebihan satu nol") +
    baris(("2025-09-20", "2025-09-21"), "Pucang Anom", "Buncis",            150000, 15000, "kelebihan satu nol") +
    baris(("2025-05-09", "2025-05-13"), "Pabean",      "Tomat",             2000,   12000, "angka depan hilang (dugaan; median hari itu 12.000)"),
    columns=["tanggal", "pasar", "komoditas_baku", "harga_asli", "harga_benar", "alasan"])

kunci = ["tanggal", "pasar", "komoditas_baku"]
daftar = KOREKSI.rename(columns={"harga_asli": "asli_daftar"})[kunci + ["asli_daftar", "harga_benar"]]

fakta["harga_asli"] = fakta["harga"]
fakta = fakta.merge(daftar, on=kunci, how="left", indicator="_k")
sudah = fakta["_k"].eq("both")

assert sudah.sum() == len(KOREKSI), "ada koreksi yang tidak menemukan barisnya"
assert (fakta.loc[sudah, "harga"] == fakta.loc[sudah, "asli_daftar"]).all(), "harga di data tidak cocok dengan harga_asli di daftar"

fakta.loc[sudah, "harga"] = fakta.loc[sudah, "harga_benar"]        # harga_benar kosong = baris dikeluarkan
fakta.loc[sudah, "harga_baku"] = fakta.loc[sudah, "harga_benar"]
fakta["dikoreksi"] = sudah
fakta = fakta.drop(columns=["_k", "asli_daftar", "harga_benar"])

PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
KOREKSI.assign(tanggal=KOREKSI["tanggal"].dt.strftime("%Y-%m-%d")).to_csv(
    PROC / "koreksi_harga.csv", index=False, encoding="utf-8-sig")
print(f"Harga dikoreksi: {int(sudah.sum())} baris dari {len(KOREKSI)} entri")

Harga dikoreksi: 11 baris dari 11 entri


## 6. Konfigurasi seleksi komoditas 
- `DIKECUALIKAN`: 22 komoditas yang tidak dipakai di Proyek 1 (kesepakatan tim). Tidak boleh masuk staging.
- 10 komoditas dashboard (`data10`) dipilih dengan aturan (bagian 7), bukan daftar tetap: data ada sepanjang periode,
  satuan tidak berganti, ketujuh pasar cukup lengkap (`MIN_PENUH`), lalu diurutkan menurut dinamika harga.

In [9]:
DIKECUALIKAN = [
    "Beras Premium", "Beras Medium Bulog", "Beras Medium Non Bulog", "Gula Pasir Lokal Curah",
    "Minyak Goreng Curah (Bening)", "Minyak Kita", "Daging Sapi Grade Atas", "Daging Sapi Grade Bawah",
    "Daging Sapi Grade Sedang", "Daging Ayam Broiler", "Telur Ayam Broiler",      # = "Telur Ayam Ras/Horn"
    "Ikan Tongkol Segar (Uk. Sedang)", "Ikan Mujair Segar (Uk. Sedang)", "Ikan Lele Segar (Ukuran Sedang)",
    "Ikan Bandeng Segar (Uk. Sedang)", 'Tepung Terigu "Segitiga Biru" Curah', "Cabe Merah Kecil-Rawit",
    "Cabe Merah Besar", "Cabe Merah Kriting", "Bawang Putih (Jenis Kating)", "Bawang Putih (Jenis Sinco)",
    "Bawang Merah (Kualitas Lokal)",
]
assert len(DIKECUALIKAN) == 22

MIN_PENUH = 85           # % minimal sel tanggal x pasar yang berharga
MIN_CV = 5               # % minimal fluktuasi harga (membuang yang nyaris datar)
MAKS_PER_KELOMPOK = 3    # batas per kelompok agar tidak didominasi satu kelompok
N_PILIH = 10
N_STABIL = 0             # komoditas stabil sebagai pembanding (0 = tidak ada)

In [10]:
MIN_PASAR = 70           # % hari berharga minimal di SETIAP pasar (tiap pasar, bukan rata-rata)

## 7. Seleksi 10 komoditas 
Tabel fakta hanya memuat harga yang teramati (baris tanpa harga dibuang). Seleksi: buang 22 pengecualian -> cek kelayakan data
-> hitung dinamika harga nasional harian (median pasar) -> urut skor, batas per kelompok.

In [11]:
kosong = fakta["harga"].isna()
print(f"Baris tanpa harga: {kosong.sum():,} dari {len(fakta):,}")
fakta_ok = fakta[~kosong].reset_index(drop=True)

assert set(DIKECUALIKAN) <= set(fakta_ok["komoditas_baku"]), "nama pengecualian tidak ada di data"
semesta = fakta_ok[~fakta_ok["komoditas_baku"].isin(DIKECUALIKAN)]
hari_total = fakta_ok["tanggal"].nunique()
n_pasar = fakta_ok["pasar"].nunique()

# kelayakan data: ada sepanjang periode, satuan tetap, pasar cukup lengkap
ringkas = (semesta.groupby("komoditas_baku")
    .agg(kelompok=("kelompok_baku", lambda s: s.mode().iat[0]),
         mulai=("tanggal", "min"), akhir=("tanggal", "max"),
         n_satuan=("satuan", "nunique"), baris=("harga", "size")))
ringkas["penuh_%"] = (ringkas["baris"] / (hari_total * n_pasar) * 100).round(1)

# kelengkapan per pasar: % hari berharga untuk setiap komoditas x pasar
per_pasar = (semesta.groupby(["komoditas_baku", "pasar"])["tanggal"].nunique()
               .unstack("pasar")
               .reindex(columns=sorted(fakta_ok["pasar"].unique()))
               .fillna(0) / hari_total * 100).round(1)
ringkas["min_pasar_%"] = per_pasar.min(axis=1)
ringkas["pasar_terlemah"] = per_pasar.idxmin(axis=1)

layak = ringkas[(ringkas["mulai"] == fakta_ok["tanggal"].min())
                & (ringkas["akhir"] == fakta_ok["tanggal"].max())
                & (ringkas["n_satuan"] == 1)
                & (ringkas["penuh_%"] >= MIN_PENUH)
                & (ringkas["min_pasar_%"] >= MIN_PASAR)].copy()      # baris baru

# dinamika harga (harga nasional harian = median pasar)
harian = (semesta[semesta["komoditas_baku"].isin(layak.index)]
          .groupby(["komoditas_baku", "tanggal"])["harga"].median().reset_index())
g = harian.groupby("komoditas_baku")["harga"]
layak["cv_%"] = (g.std() / g.mean() * 100).round(1)
layak["rentang_%"] = ((g.max() - g.min()) / g.median() * 100).round(1)

bulanan = (harian.assign(bulan=harian["tanggal"].dt.to_period("M"))
                 .groupby(["komoditas_baku", "bulan"])["harga"].mean())
mom = bulanan.groupby("komoditas_baku").pct_change().abs()
layak["bulan_berubah_%"] = (mom.gt(0.05).groupby("komoditas_baku").mean() * 100).round(1)
layak["skor"] = (layak["cv_%"].rank(pct=True) + layak["bulan_berubah_%"].rank(pct=True)) / 2

# pilih: urut skor dinamika, batas per kelompok
menarik = (layak[layak["cv_%"] >= MIN_CV]
           .sort_values("skor", ascending=False)
           .groupby("kelompok", group_keys=False).head(MAKS_PER_KELOMPOK))
utama = menarik.head(N_PILIH - N_STABIL)
stabil = layak.drop(utama.index).sort_values("cv_%").head(N_STABIL)
data10 = pd.concat([utama, stabil]).index.tolist()

assert len(data10) == N_PILIH, "kurang dari 10: turunkan MIN_PENUH/MIN_CV atau naikkan MAKS_PER_KELOMPOK"
assert not set(data10) & set(DIKECUALIKAN)
print(layak.loc[data10, ["kelompok", "penuh_%", "min_pasar_%", "pasar_terlemah", "cv_%", "bulan_berubah_%", "skor"]])
per_pasar.loc[data10].style.background_gradient(cmap="RdYlGn", vmin=0, vmax=100)

Baris tanpa harga: 107,288 dari 601,048
                                            kelompok  penuh_%  min_pasar_%  \
komoditas_baku                                                               
Tomat                                  SAYUR-SAYURAN     99.1         93.4   
Buncis                                 SAYUR-SAYURAN     97.3         81.4   
Bawang Bombay                                 BAWANG     98.2         92.3   
Wortel (Kualitas Lokal)                SAYUR-SAYURAN     96.5         75.8   
Jagung Pipilan Kering (Jenis Kecil)           JAGUNG     95.5         77.6   
Kacang Hijau                         KACANG-KACANGAN    100.0        100.0   
Udang Segar (Uk. Sedang)                        IKAN     98.6         95.0   
Telur Ayam Kampung                             TELUR    100.0         99.9   
Kacang Tanah (Jenis Besar)           KACANG-KACANGAN    100.0        100.0   
Ketela Pohon                               UBI-UBIAN     96.5         75.8   

                       

pasar,Balongsari,Genteng Baru,Kembang,Pabean,Pucang Anom,Tambahrejo,Wonokromo
komoditas_baku,,,,,,,
Tomat,93.400000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
Buncis,81.400000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
Bawang Bombay,92.300000,95.200000,100.000000,100.000000,100.000000,100.000000,100.000000
Wortel (Kualitas Lokal),75.800000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
Jagung Pipilan Kering (Jenis Kecil),93.400000,98.100000,100.000000,100.000000,100.000000,77.600000,99.700000
Kacang Hijau,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
Udang Segar (Uk. Sedang),100.000000,95.200000,99.800000,100.000000,100.000000,95.000000,100.000000
Telur Ayam Kampung,100.000000,100.000000,99.900000,100.000000,100.000000,100.000000,100.000000
Kacang Tanah (Jenis Besar),100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


## 8. Ekspor staging (10 komoditas) dan data lengkap
- `stg_harga_harian.csv`: hanya `data10`, untuk database dan dashboard. Tanggal ISO, UTF-8 tanpa BOM.
- `harga_pasar_semua_komoditas.csv`: harga per pasar untuk SEMUA komoditas.
- `seleksi_komoditas.csv`: status dan metrik tiap komoditas (dashboard / layak / tidak layak / dikecualikan).

In [12]:
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
KOLOM = ["tanggal", "pasar", "kelompok_baku", "komoditas_baku", "satuan_baku", "harga", "harga_baku", "file"]
NAMA = ["tanggal", "pasar", "kelompok", "komoditas", "satuan", "harga", "harga_baku", "sumber_file"]

def ekspor(df_, path, enc):
    out = (df_[KOLOM].set_axis(NAMA, axis=1).sort_values(["tanggal", "pasar", "kelompok", "komoditas"]))
    out["tanggal"] = out["tanggal"].dt.strftime("%Y-%m-%d")
    out.to_csv(path, index=False, encoding=enc)
    return out

stg = ekspor(fakta_ok[fakta_ok["komoditas_baku"].isin(data10)], STG / "stg_harga_harian.csv", "utf-8")
lengkap = ekspor(fakta_ok, PROC / "harga_pasar_semua_komoditas.csv", "utf-8")

tanpa.assign(tanggal=tanpa["tanggal"].dt.strftime("%Y-%m-%d")).to_csv(
    STG / "stg_hari_tanpa_data.csv", index=False, encoding="utf-8")

catatan = ringkas.join(layak[["cv_%", "rentang_%", "bulan_berubah_%", "skor"]])
catatan["status"] = "tidak layak"
catatan.loc[layak.index, "status"] = "layak"
catatan.loc[data10, "status"] = "dashboard"
catatan = catatan.reindex(ringkas.index.union(DIKECUALIKAN))
catatan.loc[DIKECUALIKAN, "status"] = "dikecualikan"
catatan.sort_values(["status", "skor"], ascending=[True, False]).to_csv(
    PROC / "seleksi_komoditas.csv", encoding="utf-8-sig")

print(f"Staging: {len(stg):,} baris | {stg['tanggal'].nunique()} tanggal | {stg['pasar'].nunique()} pasar | "
      f"{stg['komoditas'].nunique()} komoditas | {stg['kelompok'].nunique()} kelompok | {stg['sumber_file'].nunique()} file")
print("Komoditas dikecualikan yang bocor ke staging:", int(stg["komoditas"].isin(DIKECUALIKAN).sum()))
print(f"Semua komoditas: {lengkap['komoditas'].nunique()} | {len(lengkap):,} baris")
print(catatan["status"].value_counts().to_string())

Staging: 91,543 baris | 1332 tanggal | 7 pasar | 10 komoditas | 7 kelompok | 44 file
Komoditas dikecualikan yang bocor ke staging: 0
Semua komoditas: 70 | 493,760 baris
status
tidak layak     33
dikecualikan    22
dashboard       10
layak            5


## 9. Harga harian per komoditas 
Berisi SEMUA komoditas (tidak disaring seleksi Proyek 1). Rata-rata harian dari semua pasar yang mencatat harga (`harga_baku`, garam setara 500 gr). 

In [13]:
harga_komoditas = (fakta_ok
    .groupby(["tanggal", "kelompok_baku", "komoditas_baku", "satuan_baku"], as_index=False, dropna=False)
    .agg(harga=("harga_baku", "mean"), harga_min=("harga_baku", "min"),
         harga_maks=("harga_baku", "max"), n_pasar=("pasar", "nunique"))
    .rename(columns={"kelompok_baku": "kelompok", "komoditas_baku": "komoditas", "satuan_baku": "satuan"})
    .sort_values(["tanggal", "komoditas"]).reset_index(drop=True))
harga_komoditas = harga_komoditas[["tanggal", "komoditas", "satuan", "harga", "kelompok", "n_pasar", "harga_min", "harga_maks"]]
harga_komoditas.to_csv(PROC / "harga_komoditas.csv", index=False, encoding="utf-8-sig")

print(f"{len(harga_komoditas):,} baris | {harga_komoditas['tanggal'].nunique()} tanggal | "
      f"{harga_komoditas['komoditas'].nunique()} komoditas | duplikat tanggal+komoditas: "
      f"{harga_komoditas.duplicated(['tanggal', 'komoditas']).sum()}")
harga_komoditas.head()

85,456 baris | 1332 tanggal | 70 komoditas | duplikat tanggal+komoditas: 0


,tanggal,komoditas,satuan,harga,kelompok,n_pasar,harga_min,harga_maks
0,2023-01-01,Air Mineral (Aqua Galon),19 Ltr/Galon,18000.000000,AIR MINERAL,4,17000.0,19000.0
1,2023-01-01,Bawang Bombay,Kg,21714.285714,BAWANG,7,18000.0,25000.0
2,2023-01-01,Bawang Merah (Kualitas Lokal),Kg,31857.142857,BAWANG,7,28000.0,36000.0
3,2023-01-01,Bawang Putih (Jenis Kating),Kg,26714.285714,BAWANG,7,23000.0,30000.0
4,2023-01-01,Bawang Putih (Jenis Sinco),Kg,23000.000000,BAWANG,7,21000.0,25000.0


In [14]:
# Kelengkapan per pasar per bulan, rata-rata dari 10 komoditas (% hari berharga)
d10 = fakta_ok[fakta_ok["komoditas_baku"].isin(data10)].copy()
d10["bulan"] = d10["tanggal"].dt.to_period("M")
hari_bulan = fakta_ok.assign(bulan=fakta_ok["tanggal"].dt.to_period("M")).groupby("bulan")["tanggal"].nunique()

per_bulan = (d10.groupby(["bulan", "pasar", "komoditas_baku"])["tanggal"].nunique()
                .groupby(["bulan", "pasar"]).mean()
                .unstack("pasar").div(hari_bulan, axis=0).mul(100).round(0))

print(per_bulan.min().sort_values())

pasar
Balongsari       74.0
Genteng Baru     84.0
Tambahrejo       93.0
Kembang          99.0
Wonokromo        99.0
Pabean          100.0
Pucang Anom     100.0
dtype: float64


In [15]:
# Balongsari: % hari berharga per komoditas per bulan (hanya bulan di bawah 90%)
b = d10[d10["pasar"] == "Balongsari"]
tab = (b.groupby(["bulan", "komoditas_baku"])["tanggal"].nunique()
         .unstack("komoditas_baku").div(hari_bulan, axis=0).mul(100).round(0))
tab[(tab < 90).any(axis=1)]

komoditas_baku,Bawang Bombay,Buncis,Jagung Pipilan Kering (Jenis Kecil),Kacang Hijau,Kacang Tanah (Jenis Besar),Ketela Pohon,Telur Ayam Kampung,Tomat,Udang Segar (Uk. Sedang),Wortel (Kualitas Lokal)
bulan,,,,,,,,,,
2024-02,72.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2024-04,100.0,13.0,100.0,100.0,100.0,13.0,100.0,100.0,100.0,13.0
2024-09,100.0,63.0,100.0,100.0,100.0,NaN,100.0,100.0,100.0,NaN
2025-02,100.0,100.0,100.0,100.0,100.0,29.0,100.0,100.0,100.0,29.0
2026-06,100.0,13.0,13.0,100.0,100.0,100.0,100.0,13.0,100.0,100.0
